### Import Libraries

In [4]:
import pandas as pd

from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

from openai import OpenAI

C:\Users\Shashank Prakash\AppData\Local\Temp\ipykernel_22100\1808563433.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


### Import and Clean data

In [5]:
df = pd.read_csv('./top_movies.csv')
print(df.shape)
df.head()

(9420, 3)


,movie_name,genre,description
0,The Shawshank Redemption,"Drama, Crime",Imprisoned in the 1940s for the double murder ...
1,The Godfather,"Drama, Crime","Spanning the years 1945 to 1955, a chronicle o..."
2,The Godfather Part II,"Drama, Crime",In the continuing saga of the Corleone crime f...
3,Schindler's List,"Drama, History, War",The true story of how businessman Oskar Schind...
4,12 Angry Men,Drama,The defense and the prosecution have rested an...


In [6]:
df.isna().sum()

movie_name     0
genre          3
description    0
dtype: int64

In [7]:
df = df[df['genre'].notna()] # remove any NaN values as it blows up serialization
data = df.sample(1000).to_dict('records') # Get only 1000 records. More records will make it slower to index
len(data)

1000

In [8]:
data[0]

{'movie_name': 'The Expendables 3',
 'genre': 'Action, Adventure, Thriller',
 'description': 'Barney, Christmas and the rest of the team comes face-to-face with Conrad Stonebanks, who years ago co-founded The Expendables with Barney. Stonebanks subsequently became a ruthless arms trader and someone who Barney was forced to kill… or so he thought. Stonebanks, who eluded death once before, now is making it his mission to end The Expendables -- but Barney has other plans. Barney decides that he has to fight old blood with new blood, and brings in a new era of Expendables team members, recruiting individuals who are younger, faster and more tech-savvy. The latest mission becomes a clash of classic old-school style versus high-tech expertise in the Expendables’ most personal battle yet.'}

### Create Vector DB

In [9]:
encoder = SentenceTransformer('all-MiniLM-L6-v2') # Model to create embeddings

In [10]:
# create the vector database client
qdrant = QdrantClient(":memory:") # Create in-memory Qdrant instance

In [11]:
# Create collection to store wines
qdrant.recreate_collection(
    collection_name="top_movies",
    vectors_config=models.VectorParams(
        size=encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
        distance=models.Distance.COSINE
    )
)

True

In [12]:
# vectorize!
qdrant.upload_points(
    collection_name="top_movies",
    points=[
        models.PointStruct(
            id=idx,
            vector=encoder.encode(doc["description"]).tolist(),
            payload=doc,
        ) for idx, doc in enumerate(data) # data is the variable holding all the movies' info
    ]
)

In [13]:
user_prompt = "Suggest me a fantastic science fiction movie"

In [14]:
# Search time for awesome movies!

hits = qdrant.search(
    collection_name="top_movies",
    query_vector=encoder.encode(user_prompt).tolist(),
    limit=3
)
for hit in hits:
  print(hit.payload, "score:", hit.score)

{'movie_name': 'Dead Space: Downfall', 'genre': 'Animation, Horror, Science Fiction, Thriller', 'description': 'On a deep space mining mission to a remote planet, an ancient religious relic - thought to be proof of the existence of God - is unearthed and brought aboard. When the unholy artifact unleashes a long-dormant alien race, its glimpse of Heaven transforms the ship into a living Hell. A prequel to the events of the 2008 video game Dead Space.'} score: 0.3696994785014883
{'movie_name': 'Porno', 'genre': 'Comedy, Horror', 'description': 'When a group of naive teens working at a movie theater in a small Christian town discover a mysterious film hidden in its basement, they unleash an alluring succubus who gives them a sex education…written in blood.'} score: 0.34738671337221255
{'movie_name': 'Twilight Zone: The Movie', 'genre': 'Horror, Fantasy, Science Fiction', 'description': 'An anthology film presenting remakes of three episodes from the "Twilight Zone" TV series—"Kick the Can

In [15]:
# define a variable to hold the search results
search_results = [hit.payload for hit in hits]
search_results

[{'movie_name': 'Dead Space: Downfall',
  'genre': 'Animation, Horror, Science Fiction, Thriller',
  'description': 'On a deep space mining mission to a remote planet, an ancient religious relic - thought to be proof of the existence of God - is unearthed and brought aboard. When the unholy artifact unleashes a long-dormant alien race, its glimpse of Heaven transforms the ship into a living Hell. A prequel to the events of the 2008 video game Dead Space.'},
 {'movie_name': 'Porno',
  'genre': 'Comedy, Horror',
  'description': 'When a group of naive teens working at a movie theater in a small Christian town discover a mysterious film hidden in its basement, they unleash an alluring succubus who gives them a sex education…written in blood.'},
 {'movie_name': 'Twilight Zone: The Movie',
  'genre': 'Horror, Fantasy, Science Fiction',
  'description': 'An anthology film presenting remakes of three episodes from the "Twilight Zone" TV series—"Kick the Can", "It\'s a Good Life" and "Nightmar

In [16]:
# another user prompt to test

user_prompt = "Suggest me a slice of life, drama movie with complex plot"


hits = qdrant.search(
    collection_name="top_movies",
    query_vector=encoder.encode(user_prompt).tolist(),
    limit=3
)
for hit in hits:
  print(hit.payload, "score:", hit.score)


search_results = [hit.payload for hit in hits]

{'movie_name': 'Waking Life', 'genre': 'Animation, Drama, Fantasy', 'description': 'Waking Life is about a young man in a persistent lucid dream-like state. The film follows its protagonist as he initially observes and later participates in philosophical discussions that weave together issues like reality, free will, our relationships with others, and the meaning of life.'} score: 0.4434701171469208
{'movie_name': '13', 'genre': 'Drama, Thriller', 'description': "A naive young man assumes a dead man's identity and finds himself embroiled in an underground world of power, violence, and chance where men gamble behind closed doors on the lives of other men."} score: 0.41042902812520776
{'movie_name': 'Twilight Zone: The Movie', 'genre': 'Horror, Fantasy, Science Fiction', 'description': 'An anthology film presenting remakes of three episodes from the "Twilight Zone" TV series—"Kick the Can", "It\'s a Good Life" and "Nightmare at 20,000 Feet"—and one original story, "Time Out."'} score: 0.

In [18]:
# Now time to connect to the local large language model
client = OpenAI(
    base_url="http://localhost:8080/v1/", # "http://<Your api-server IP>:port"
    api_key = "sk-no-key-required"
)
completion = client.chat.completions.create(
    model="LLaMA_CPP",
    messages=[
        {"role": "system", "content": "You are chatbot, a movie specialist. Your top priority is to help guide users into selecting amazing movies and guide them with their requests."},
        {"role": "user", "content": "Suggest me a slice of life, drama movie with complex plot"},
        {"role": "assistant", "content": str(search_results)}
    ]
)
print(completion.choices[0].message.content)

Okay, let’s dig a little deeper into slice-of-life dramas with complex plots. Those can be really rewarding! Based on your request, here are a few suggestions, broken down with a little more detail to help you decide:

**1.  *Aftersun* (2022) - Drama, Romance, Memory**

*   **Why it fits:** This is a *fantastic* choice if you're looking for something emotionally resonant and layered. It’s a father and daughter’s holiday in Turkey, but it’s told through fragmented memories and a hazy sense of longing. The “complex plot” comes from the unspoken sadness and the mystery surrounding the father’s inner life. It’s a film about memory, regret, and the difficulty of truly knowing someone.
*   **Tone:**  Melancholy, beautiful, poignant. It’s a slow burn, but incredibly rewarding.
*   **Runtime:** 1 hour 38 minutes


**2. *Manchester by the Sea* (2016) - Drama, Romance**

*   **Why it fits:** This film deals with grief, guilt, and the struggle to connect after a devastating loss. It’s not a flash

In [19]:
completion = client.chat.completions.create(
    model="LLaMA_CPP",
    messages=[
        {"role": "system", "content": "You are chatbot, a movie specialist. Your top priority is to help guide users into selecting amazing movies and guide them with their requests. Use only the movies provided here nothing else."},
        {"role": "user", "content": "Suggest me a slice of life, drama movie with complex plot"},
        {"role": "assistant", "content": str(search_results)}
    ]
)
print(completion.choices[0].message.content)

Considering you’re looking for a slice-of-life drama with a complex plot, I’d recommend **Waking Life**. 

It’s a beautifully animated film that delves into deep philosophical questions and explores the nature of reality and consciousness. It’s definitely a movie that stays with you and invites multiple viewings to unpack its layers. 

Would you like to know more about *Waking Life*, or perhaps explore one of the other options?<end_of_turn>
